### Preprocessing and features engineering

Now we have a clean dataset with tennis matches starting from **2005-07-04** until **2026** and we need to preprocess it, by reducing the number of features and creating new ones to improve the model performance. Afterwards, we will split the dataset into training and test sets and start the model training.

In [159]:
import pandas as pd
import numpy as np

RAW_TRAINING_PATH = "../data/training/tennis_training.xlsx"
RAW_TESTING_PATH = "../data/testing/tennis_testing.xlsx"

raw_train_df = pd.read_excel(RAW_TRAINING_PATH)
raw_test_df = pd.read_excel(RAW_TESTING_PATH)
raw_split_index = len(raw_train_df)

# Prepare both raw splits together to keep chronological historical features consistent.
df_raw = pd.concat([raw_train_df, raw_test_df], ignore_index=True)
df_raw.head()

,Date,Series,Court,Surface,Round,Tournament,Best of,Winner,Loser,WRank,...,W3,L3,W4,L4,W5,L5,B365W,B365L,Wsets,Lsets
0,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,3.0,Robredo T.,Tabara M.,20.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.10,6.00,2.0,0.0
1,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,3.0,Vinciguerra A.,Ryderstedt M.,917.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.10,1.66,2.0,0.0
2,2005-07-04,International,Outdoor,Clay,1st Round,Allianz Suisse Open,3.0,Verdasco F.,Pospisil J.,59.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0
3,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,3.0,Ginepri R.,Oudsema S.,103.0,...,6.0,0.0,NaN,NaN,NaN,NaN,1.12,5.50,2.0,1.0
4,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,3.0,Spadea V.,Popp A.,49.0,...,6.0,2.0,NaN,NaN,NaN,NaN,2.50,1.50,2.0,1.0


Firstly, we must randomly swap the winner and loser players in each match, to avoid the model learning about choosing always the first player as the winner, causing a data leakage.

In [160]:
df_prepared = df_raw.copy()

# Randomly assign the winner to Player 1 or Player 2.
rng = np.random.default_rng(42)
winner_is_player_1 = rng.random(len(df_prepared)) < 0.5

df_prepared["Player_1"] = np.where(
    winner_is_player_1, df_raw["Winner"], df_raw["Loser"]
)
df_prepared["Player_2"] = np.where(
    winner_is_player_1, df_raw["Loser"], df_raw["Winner"]
)

# Target: 1 if Player 1 wins, otherwise 0.
df_prepared["y"] = winner_is_player_1.astype(int)

# Align player statistics with the randomized player positions.
for name, winner_column, loser_column in [
    ("Rank", "WRank", "LRank"),
    ("Pts", "WPts", "LPts"),
    ("Odds", "B365W", "B365L"),
    ("Sets", "Wsets", "Lsets"),
    ("games1", "W1", "L1"),
    ("games2", "W2", "L2"),
    ("games3", "W3", "L3"),
    ("games4", "W4", "L4"),
    ("games5", "W5", "L5"),
]:
    df_prepared[f"{name}_1"] = np.where(
        winner_is_player_1,
        df_raw[winner_column],
        df_raw[loser_column],
    )
    df_prepared[f"{name}_2"] = np.where(
        winner_is_player_1,
        df_raw[loser_column],
        df_raw[winner_column],
    )
    
df_prepared.head()

,Date,Series,Court,Surface,Round,Tournament,Best of,Winner,Loser,WRank,...,games1_1,games1_2,games2_1,games2_2,games3_1,games3_2,games4_1,games4_2,games5_1,games5_2
0,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,3.0,Robredo T.,Tabara M.,20.0,...,5.0,7.0,0.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2005-07-04,International,Outdoor,Clay,1st Round,Swedish Open,3.0,Vinciguerra A.,Ryderstedt M.,917.0,...,6.0,3.0,6.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2005-07-04,International,Outdoor,Clay,1st Round,Allianz Suisse Open,3.0,Verdasco F.,Pospisil J.,59.0,...,2.0,6.0,4.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,3.0,Ginepri R.,Oudsema S.,103.0,...,2.0,6.0,7.0,6.0,0.0,6.0,NaN,NaN,NaN,NaN
4,2005-07-04,International,Outdoor,Grass,1st Round,Hall of Fame Championships,3.0,Spadea V.,Popp A.,49.0,...,4.0,6.0,6.0,3.0,6.0,2.0,NaN,NaN,NaN,NaN


#### Numerical features

We can start by creating simple engineered features, such as the differences between the players' ranks, points, and betting market odds.

We can also use logarithmic differences. This is useful because these variables are not linear.

**Example 1**

- Rank 1 vs 10
- Rank 101 vs 110

The absolute difference is the same (9), but the first difference is much more meaningful than the second one.

**Example 2**

- Points 9000 vs 8000
- Points 1200 vs 1000

Same considerations as example 1.

**Example 3**

- Odds 1.20 vs 1.50
- Odds 4.00 vs 4.30

Same absolute difference, but the second one is much more uncertain than the first one.

For betting odds, we use a log-ratio instead of a raw difference. This is closer to the market implied probability: positive values mean Player 1 is favored by the market, negative values mean Player 2 is favored.

Moreover, we will consider features distribution to limit outliers and avoid extreme values that could dominate the model training. We use the `np.clip` function to limit the values of the features in a specific range, which is established in the 1-99 percentile range of the feature distribution.

In [161]:
p1_rank, p2_rank = df_prepared["Rank_1"], df_prepared["Rank_2"]
p1_pts, p2_pts = df_prepared["Pts_1"], df_prepared["Pts_2"]
p1_odds, p2_odds = df_prepared["Odds_1"], df_prepared["Odds_2"]

# log1p(x) = log(1 + x), avoiding log(0)
# clip: avoids extreme values that could dominate the model training
# feature distribution: limit values to the 1-99 percentile
# nanquantile: ignore NaN values

rank_diff = p1_rank - p2_rank
rank_diff_quantiles = np.nanquantile(rank_diff, [0.01, 0.99])
df_prepared["Rank_Diff"] = np.clip(rank_diff, rank_diff_quantiles[0], rank_diff_quantiles[1])


# df_prepared["Points_Diff"] = np.log1p(p1_pts) - np.log1p(p2_pts)
points_diff = p1_pts - p2_pts
points_diff_quantiles = np.nanquantile(points_diff, [0.01, 0.99])
df_prepared["Points_Diff"] = np.clip(points_diff, points_diff_quantiles[0], points_diff_quantiles[1])


# Market log-odds ratio: positive means Player 1 is favored by the market.
# This is more informative than a raw odds difference because betting odds are multiplicative.
valid_odds = (p1_odds > 0) & (p2_odds > 0)
df_prepared["Odds_Diff"] = np.nan
df_prepared.loc[valid_odds, "Odds_Diff"] = np.log(
    p2_odds[valid_odds] / p1_odds[valid_odds]
)

We can convert **Best of** feature into a binary feature **Best of 5**, which is 1 if the match is best of 5 sets and 0 otherwise.

In [162]:
df_prepared["Best_of_5"] = (df_raw["Best of"] == 5).astype(int)

We use the cleaned players dataset to create an age-difference feature.

The players file already contains match-compatible keys such as `Ruud C.` and keeps duplicated keys.
Here we only select a date of birth when exactly one candidate gives a plausible age at match date.


In [163]:
PLAYERS_PATH = "../data/training/tennis_players.xlsx"
players_df = pd.read_excel(PLAYERS_PATH)
players_df["dob"] = pd.to_datetime(players_df["dob"], errors="coerce")

MIN_PLAYER_AGE = 16
MAX_PLAYER_AGE = 45

# The player keys are already cleaned in notebook 01.
player_dob_candidates = (
    players_df
    .dropna(subset=["player_key", "dob"])
    .groupby("player_key")["dob"]
    .apply(list)
    .to_dict()
)

def get_plausible_age(player_name, match_date):
    # Return an age only when one DOB candidate is plausible for this match date.
    plausible_ages = []

    for dob in player_dob_candidates.get(player_name, []):
        age = (match_date - dob).days / 365.25
        if MIN_PLAYER_AGE <= age <= MAX_PLAYER_AGE:
            plausible_ages.append(age)

    # If multiple candidates remain, keep NaN instead of guessing.
    if len(plausible_ages) == 1:
        return plausible_ages[0]

    return np.nan

age_diff = []

for row in df_prepared.itertuples(index=False):
    age_1 = get_plausible_age(row.Player_1, row.Date)
    age_2 = get_plausible_age(row.Player_2, row.Date)
    age_diff.append(age_1 - age_2)

df_prepared["Age_Diff"] = age_diff

print("Age_Diff coverage:", f"{df_prepared['Age_Diff'].notna().mean():.2%}")
print("Extreme age differences > 20 years:", (df_prepared["Age_Diff"].abs() > 20).sum())


Age_Diff coverage: 75.38%
Extreme age differences > 20 years: 9


In [164]:
numerical_features = [
    "Rank_Diff",
    "Points_Diff",
    "Odds_Diff",
    "Best_of_5",
    "Age_Diff",
]

#### Advanced features

We can improve the model performance by creating *advanced* features about players ELO ratings, fatigue and head to head statistics.

Firstly, we sort the dataset by date to avoid data leakage

In [165]:
# Elo must be calculated in chronological order.
df_prepared["Date"] = pd.to_datetime(
    df_prepared["Date"],
    errors="raise",
)

# sort dataframe by date to avoid data leakage when calculating Elo ratings
df_prepared = (
    df_prepared
    .sort_values("Date", kind="stable")
    .reset_index(drop=True) # reset index after sorting
)

#### Fatigue

In [166]:
from collections import defaultdict, deque

FATIGUE_WINDOW_DAYS = 10
FATIGUE_DECAY_DAYS = 3.0
DEFAULT_REST_DAYS = 30

recent_matches = defaultdict(list)
fatigue_diff = []


def player_fatigue(player, match_date):
    """Return recent match load before the current match."""
    fatigue = 0.0

    for previous_date in recent_matches[player]:
        days_since_match = (match_date - previous_date).days

        if 0 < days_since_match <= FATIGUE_WINDOW_DAYS:
            fatigue += np.exp(-days_since_match / FATIGUE_DECAY_DAYS)

    return fatigue


def player_rest_days(player, match_date):
    """Return days since previous match, using a neutral value for new players."""
    if not recent_matches[player]:
        return DEFAULT_REST_DAYS
    return (match_date - recent_matches[player][-1]).days


def update_fatigue(player_1, player_2, match_date):
    """Store pre-match fatigue difference, then update recent match dates."""
    fatigue_1 = player_fatigue(player_1, match_date)
    fatigue_2 = player_fatigue(player_2, match_date)
    rest_days_1 = player_rest_days(player_1, match_date)
    rest_days_2 = player_rest_days(player_2, match_date)

    # Store pre-match fatigue difference to avoid data leakage.
    fatigue_diff.append(fatigue_1 - fatigue_2)

    # Update match history only after computing the feature.
    recent_matches[player_1].append(match_date)
    recent_matches[player_2].append(match_date)

    return rest_days_1, rest_days_2

#### ELO ratings

Each player starts with an ELO rating of 1500. First, we compute the expected probability that Player 1 wins:

$$ E_1 = \frac{1}{1 + 10^{\frac{R_2 - R_1}{400}}} $$

Then, after the match, we update the rating using:

$$ \Delta R = K \cdot (S_1 - E_1) $$

where:

- $R_1, R_2$ are the pre-match ELO ratings of Player 1 and Player 2
- $E_1$ is the expected probability that Player 1 wins
- $S_1$ is the actual result: 1 if Player 1 wins, 0 otherwise
- $K$ controls how strongly ratings are updated

$K$ is dynamically reduced as a player accumulates matches, because the rating becomes more reliable:

$$ K = \max( \; \texttt{K\_MIN}, \frac{\texttt{K\_BASE}}{1 + \frac{\texttt{matches\_played}}{100}} \; ) $$

with $\texttt{K\_MIN} = 8, \; \texttt{K\_BASE} = 32$.

Following Tennis Abstract, surface ELO is used as a 50/50 blend between the player's overall ELO and raw surface ELO. The article also discusses absence handling, but the exact penalty is not fully specified, so here long absences only increase the K factor slightly instead of subtracting rating points directly.

In [167]:
K_MIN = 8.0
K_BASE = 32.0
BASE_ELO = 1500.0

SURFACE_BLEND_WEIGHT = 0.5

ABSENCE_THRESHOLD_DAYS = 90
ABSENCE_K_MULTIPLIER = 1.25

USE_ABSENCE_RATING_PENALTY = True
MAX_ABSENCE_RATING_PENALTY = 100.0
ABSENCE_PENALTY_PER_DAY = 0.25

# Count matches played for dynamic K-factor
matches_played = {}
matches_played_surface = {}

# Overall Elo
elo_ratings = {}
elo_diff = []
elo_prob = []
elo_experience_diff = []

# Surface Elo
surface_elo_ratings = {}
surface_elo_diff = []
surface_elo_prob = []


def dynamic_k(player):
    """Return a larger K for new players and a smaller K for experienced players."""
    played = matches_played.get(player, 0)
    return max(K_MIN, K_BASE / (1 + played / 100))


def dynamic_k_surface(player, surface):
    """Return a larger K for new players on a specific surface."""
    played = matches_played_surface.get((player, surface), 0)
    return max(K_MIN, K_BASE / (1 + played / 100))


def absence_k_multiplier(rest_days):
    """Increase K after a long absence because the rating is less certain."""
    if rest_days >= ABSENCE_THRESHOLD_DAYS:
        return ABSENCE_K_MULTIPLIER
    return 1.0


def absence_rating_penalty(rest_days):
    """Optionally reduce rating after a long absence."""
    if not USE_ABSENCE_RATING_PENALTY:
        return 0.0

    if rest_days < ABSENCE_THRESHOLD_DAYS:
        return 0.0

    penalty = (rest_days - ABSENCE_THRESHOLD_DAYS) * ABSENCE_PENALTY_PER_DAY

    return min(MAX_ABSENCE_RATING_PENALTY, penalty)


def blended_surface_rating(overall_rating, surface_rating):
    """Blend overall Elo and surface Elo."""
    return (
        (1 - SURFACE_BLEND_WEIGHT) * overall_rating
        + SURFACE_BLEND_WEIGHT * surface_rating
    )


def expected_score(rating_1, rating_2):
    """Return the expected probability that player 1 beats player 2."""
    return 1 / (1 + 10 ** ((rating_2 - rating_1) / 400))


def update_elo_ratings(player_1, player_2, surface, y, rest_days_1, rest_days_2):
    # Pre-match overall Elo
    raw_elo_player1 = elo_ratings.get(player_1, BASE_ELO)
    raw_elo_player2 = elo_ratings.get(player_2, BASE_ELO)

    # Apply temporary absence penalty only for prediction/update computation
    elo_player1 = raw_elo_player1 - absence_rating_penalty(rest_days_1)
    elo_player2 = raw_elo_player2 - absence_rating_penalty(rest_days_2)

    # Pre-match raw surface Elo
    raw_surface_elo_player1 = surface_elo_ratings.get((player_1, surface), BASE_ELO)
    raw_surface_elo_player2 = surface_elo_ratings.get((player_2, surface), BASE_ELO)

    surface_elo_player1 = raw_surface_elo_player1 - absence_rating_penalty(rest_days_1)
    surface_elo_player2 = raw_surface_elo_player2 - absence_rating_penalty(rest_days_2)

    # Blended surface Elo: 50% overall + 50% surface
    blended_surface_elo_player1 = blended_surface_rating(
        elo_player1,
        surface_elo_player1,
    )

    blended_surface_elo_player2 = blended_surface_rating(
        elo_player2,
        surface_elo_player2,
    )

    # Expected probabilities
    expected_1 = expected_score(elo_player1, elo_player2)

    expected_surface_1 = expected_score(
        blended_surface_elo_player1,
        blended_surface_elo_player2,
    )

    # Store pre-match features
    elo_diff.append(elo_player1 - elo_player2)
    elo_prob.append(expected_1)

    surface_elo_diff.append(
        blended_surface_elo_player1 - blended_surface_elo_player2
    )
    surface_elo_prob.append(expected_surface_1)

    elo_experience_diff.append(
        np.log1p(matches_played.get(player_1, 0))
        - np.log1p(matches_played.get(player_2, 0))
    )

    # Player-specific K values
    k1 = dynamic_k(player_1) * absence_k_multiplier(rest_days_1)
    k2 = dynamic_k(player_2) * absence_k_multiplier(rest_days_2)

    # Update overall Elo with separate K
    elo_ratings[player_1] = raw_elo_player1 + k1 * (y - expected_1)
    elo_ratings[player_2] = raw_elo_player2 - k2 * (y - expected_1)

    # Surface-specific K values
    k_surface_1 = dynamic_k_surface(player_1, surface) * absence_k_multiplier(rest_days_1)
    k_surface_2 = dynamic_k_surface(player_2, surface) * absence_k_multiplier(rest_days_2)

    # Update raw surface Elo with separate K
    surface_elo_ratings[(player_1, surface)] = (
        raw_surface_elo_player1 + k_surface_1 * (y - expected_surface_1)
    )

    surface_elo_ratings[(player_2, surface)] = (
        raw_surface_elo_player2 - k_surface_2 * (y - expected_surface_1)
    )

    # Update match counts only after computing all pre-match features
    matches_played[player_1] = matches_played.get(player_1, 0) + 1
    matches_played[player_2] = matches_played.get(player_2, 0) + 1

    matches_played_surface[(player_1, surface)] = (
        matches_played_surface.get((player_1, surface), 0) + 1
    )

    matches_played_surface[(player_2, surface)] = (
        matches_played_surface.get((player_2, surface), 0) + 1
    )

#### Recent form

We can store the last 5 matches played by each player using a queue and compute the **recent form** as the average of the last 5 matches played.

E.g. 
- $[1, 0, 1, 0, 1] \implies 0.6$

- $[1] \implies 1.0$

- $[0, 0, 1] \implies 0.33$

We can also try a weighted recent form, assigning an higher weight to most important matches as

$$ \texttt{weighted\_recent\_form} = \texttt{result} \cdot \frac{\texttt{opponent\_ELO}}{\texttt{BASE\_ELO}} $$

In [168]:
RECENT_FORM_WINDOW = 5

weighted_recent_results = defaultdict(lambda: deque(maxlen=RECENT_FORM_WINDOW))
recent_form_diff = []

weighted_recent_surface_results = defaultdict(lambda: deque(maxlen=5))
recent_surface_form_diff = []

def update_recent_results(player_1, player_2, surface, y):
    
    # compute the mean of the last RECENT_FORM_WINDOW matches
    form_1 = np.mean(weighted_recent_results[player_1]) if weighted_recent_results[player_1] else 0.5
    form_2 = np.mean(weighted_recent_results[player_2]) if weighted_recent_results[player_2] else 0.5
    
    form_surface_1 = np.mean(weighted_recent_surface_results[(player_1, surface)]) if weighted_recent_surface_results[(player_1, surface)] else 0.5
    form_surface_2 = np.mean(weighted_recent_surface_results[(player_2, surface)]) if weighted_recent_surface_results[(player_2, surface)] else 0.5
    
    # save pre-match recent form, avoiding data leakage
    recent_form_diff.append(form_1 - form_2)
    recent_surface_form_diff.append(form_surface_1 - form_surface_2)
    
    # weighted_res = res * opponent_elo / base_elo
    weighted_recent_results[player_1].append(y * (elo_ratings.get(player_2, BASE_ELO) / BASE_ELO))
    weighted_recent_results[player_2].append((1 - y) * (elo_ratings.get(player_1, BASE_ELO) / BASE_ELO))
    
    weighted_recent_surface_results[(player_1, surface)].append(y * (surface_elo_ratings.get((player_2, surface), BASE_ELO) / BASE_ELO))
    weighted_recent_surface_results[(player_2, surface)].append((1 - y) * (surface_elo_ratings.get((player_1, surface), BASE_ELO) / BASE_ELO))



We can also compute the **dominance form**, by considering the number of games won by each player in the last matches, instead of just the match result.

In [169]:
"""Compute number of games won by each player """
games_1 = df_prepared[["games1_1", "games2_1", "games3_1", "games4_1", "games5_1"]]
games_2 = df_prepared[["games1_2", "games2_2", "games3_2", "games4_2", "games5_2"]]

df_prepared["Games_P1"] = games_1.sum(axis=1, skipna=True)
df_prepared["Games_P2"] = games_2.sum(axis=1, skipna=True)

print(df_prepared[["Games_P1", "Games_P2", "Wsets", "Lsets", "y"]].head())

   Games_P1  Games_P2  Wsets  Lsets  y
0       5.0      13.0    2.0    0.0  0
1      12.0       4.0    2.0    0.0  1
2       6.0      12.0    2.0    0.0  0
3       9.0      18.0    2.0    1.0  0
4      16.0      11.0    2.0    1.0  1


In [170]:
weighted_dominance_results = defaultdict(lambda: deque(maxlen=RECENT_FORM_WINDOW))
dominance_form_diff = []

def update_dominance_form(player_1, player_2, games_won_1, games_total_1):

    dom_1 = np.mean(weighted_dominance_results[player_1]) if weighted_dominance_results[player_1] else 0.5
    dom_2 = np.mean(weighted_dominance_results[player_2]) if weighted_dominance_results[player_2] else 0.5

    # save pre-match, avoiding leakage
    dominance_form_diff.append(dom_1 - dom_2)

    # ratio of games won on this match
    games_ratio_1 = games_won_1 / games_total_1 if games_total_1 > 0 else 0.5
    games_ratio_2 = 1 - games_ratio_1

    weighted_dominance_results[player_1].append(games_ratio_1)
    weighted_dominance_results[player_2].append(games_ratio_2)

#### Head to head statistics

Head-to-head features count previous wins between the same two players, globally and on the current surface. Values are stored before updating the current match to avoid data leakage.

In [171]:
h2h_results = defaultdict(lambda: {"wins": 0, "losses": 0})
h2h_surface_results = defaultdict(lambda: {"wins": 0, "losses": 0})

h2h_diff = []
h2h_surface_diff = []


def update_h2h(player_1, player_2, surface, y):
    """Store pre-match head-to-head differences, then update matchup history."""
    key_1 = (player_1, player_2)
    key_2 = (player_2, player_1)

    surface_key_1 = (player_1, player_2, surface)
    surface_key_2 = (player_2, player_1, surface)

    # Store pre-match H2H difference to avoid data leakage.
    h2h_diff.append(
        h2h_results[key_1]["wins"] - h2h_results[key_1]["losses"]
    )

    h2h_surface_diff.append(
        h2h_surface_results[surface_key_1]["wins"]
        - h2h_surface_results[surface_key_1]["losses"]
    )

    # Update H2H only after computing the feature.
    if y == 1:
        h2h_results[key_1]["wins"] += 1
        h2h_results[key_2]["losses"] += 1

        h2h_surface_results[surface_key_1]["wins"] += 1
        h2h_surface_results[surface_key_2]["losses"] += 1
    else:
        h2h_results[key_1]["losses"] += 1
        h2h_results[key_2]["wins"] += 1

        h2h_surface_results[surface_key_1]["losses"] += 1
        h2h_surface_results[surface_key_2]["wins"] += 1

In [172]:
for row in df_prepared.itertuples(index=False):
    player_1, player_2 = row.Player_1, row.Player_2
    match_date, surface = row.Date, row.Surface
    y = row.y
    
    rest_days_1, rest_days_2 = update_fatigue(
        player_1,
        player_2,
        match_date,
    )
    
    update_elo_ratings(
        player_1,
        player_2,
        surface,
        y,
        rest_days_1,
        rest_days_2,
    )

    update_recent_results(player_1, player_2, surface, y)
    
    games_won_P1 = row.Games_P1
    games_total = row.Games_P1 + row.Games_P2
    
    update_dominance_form(player_1, player_2, games_won_P1, games_total)

    update_h2h(player_1, player_2, surface, y)

df_prepared["Recent_Form_Diff"] = recent_form_diff

df_prepared["Dominance_Form_Diff"] = dominance_form_diff

df_prepared["Recent_Surface_Form_Diff"] = recent_surface_form_diff

df_prepared["Fatigue_Diff"] = fatigue_diff

# --- ELO ---
elo_diff_quantiles = np.nanquantile(elo_diff, [0.01, 0.99])
df_prepared["Elo_Diff"] = np.clip(elo_diff, elo_diff_quantiles[0], elo_diff_quantiles[1])

surface_elo_diff_quantiles = np.nanquantile(surface_elo_diff, [0.01, 0.99])
df_prepared["Surface_Elo_Diff"] = np.clip(surface_elo_diff, surface_elo_diff_quantiles[0], surface_elo_diff_quantiles[1])

# --- H2H ---
df_prepared["H2H_Diff"] = h2h_diff
df_prepared["H2H_Surface_Diff"] = h2h_surface_diff


numerical_features += [
    "Recent_Form_Diff",
    "Recent_Surface_Form_Diff",
    "Fatigue_Diff",
    "Dominance_Form_Diff",

    "Elo_Diff",

    "Surface_Elo_Diff",

    "H2H_Diff",
    "H2H_Surface_Diff",
]

In [173]:
"""Print computed ELO ratings"""

# sort elo ratings and print
sorted(elo_ratings.items(), key=lambda x: x[1], reverse=True)
# filter "Sinner J." surface ratigs
# for surface in ["Hard", "Clay", "Grass"]:
    # print(f"Sinner J. {surface} Elo: {surface_elo_ratings.get(('Sinner J.', surface), STARTING_ELO)}")

[('Sinner J.', 2065.4293317394845),
 ('Alcaraz C.', 2027.2759271969308),
 ('Djokovic N.', 2010.74371888346),
 ('Federer R.', 1999.4151957810445),
 ('Nadal R.', 1973.6138673135113),
 ('Zverev A.', 1901.2018091427979),
 ('Del Potro J.M.', 1875.0326917059042),
 ('Medvedev D.', 1844.178909073414),
 ('Soderling R.', 1839.3344728234752),
 ('Draper J.', 1819.86112885042),
 ('Fils A.', 1811.7829458727294),
 ('De Minaur A.', 1805.865348217243),
 ('Fritz T.', 1802.7838646020753),
 ('Roddick A.', 1791.9068392788113),
 ('Ruud C.', 1782.8282247531772),
 ('Paul T.', 1782.8119653525634),
 ('Rune H.', 1776.532552357832),
 ('Shelton B.', 1775.7151359457764),
 ('Auger Aliassime F.', 1771.7412283719964),
 ('Raonic M.', 1767.9875522574546),
 ('Musetti L.', 1767.386035851148),
 ('Berdych T.', 1765.7489476469198),
 ('Kyrgios N.', 1764.3797301563006),
 ('Bautista R.', 1754.7191920959044),
 ('Mensik J.', 1753.1863561399657),
 ('Lehecka J.', 1752.0190758320196),
 ('Rublev A.', 1750.5247016997516),
 ('Tien L.',

#### Categorical features

We create an imputer to manage missing values, using a **most frequent** strategy and **One hot encoding** to convert categorical features into numerical ones.

In [174]:
categorical_features = [
    "Series",
    "Court",
    "Surface",
    "Round"
]

We manage high cardinality categorical features, such as **Tournament**, by keeping only the most frequent values and grouping the others into a single category called **Other**.

In [175]:
high_cardinality_categorical_features = [
    "Tournament",
]


In [176]:
df_prepared["Elo_Diff_x_EarlyRound"] = (
    df_prepared["Elo_Diff"] *
    df_prepared["Round"].isin(["1st Round", "2nd Round", "3rd Round"]).astype(int)
)

numerical_features += ["Elo_Diff_x_EarlyRound"]

#### Pipeline

Finally, we can create an **imputer** to handle missing values in the dataset. We will use this later, after the splitting phase, to avoid data leakage.

In [177]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    ))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# Create a preprocessor that combines both numeric and categorical pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features),
])

We will use it later during the model training.

#### Splitting and save the dataset

Now we can split the dataset into training and test sets

In [178]:
features = [
    "Date",
    "Player_1", "Player_2"
] + numerical_features + categorical_features + high_cardinality_categorical_features + ["y"]

df_prepared = df_prepared[features].copy()

train_df = df_prepared.iloc[:raw_split_index].copy()
test_df = df_prepared.iloc[raw_split_index:].copy()


And save them as new `xlsx` datasets, to be used in the next notebook for model training and evaluation.

In [179]:
from pathlib import Path

TRAINING_PATH = Path("../data/prepared/tennis_training.xlsx")
TESTING_PATH = Path("../data/prepared/tennis_testing.xlsx")

TRAINING_PATH.parent.mkdir(parents=True, exist_ok=True)
TESTING_PATH.parent.mkdir(parents=True, exist_ok=True)

train_df.to_excel(TRAINING_PATH, index=False)
test_df.to_excel(TESTING_PATH, index=False)